### Вариант 2 (рабочий): из HTML читаем

In [1]:
from bs4 import BeautifulSoup
import json
import re

html_file = "/Users/konstantin/Documents/Заказ оформлен.html"
with open(html_file, "r", encoding="utf-8") as file:
    soup = BeautifulSoup(file, "html.parser")
    
raw_text = ''
for script in soup.find_all("script"):
    if len(script.text) > 100 and script.text.startswith("window.__REACT_QUERY_STATE__"):
        raw_text = script.text[29:]

# чиcтим чтобы влезло в JSON
match = re.search(r'({.*})', raw_text, re.DOTALL)
if not match:
    raise ValueError("No JSON-like block found in the file.")
json_like_text = match.group(1)
cleaned_json_text = (
    json_like_text
    .replace("undefined", "null")  # JS -> JSON
    .replace("True", "true")
    .replace("False", "false")
)

try:
    data = json.loads(cleaned_json_text)
except json.JSONDecodeError as e:
    raise ValueError(f"JSON parsing error: {e}")

# json[json.find("Рис")-50:json.find("Рис")+100]
titles = [x['name'] for x in data['queries'][1]['state']['data']['calculation']['items']]

for i, title in enumerate(titles, 1):
    print(f"{i}. {title}")


1. Мороженое протеиновое Bombbar фисташковое без сахара в вафельном стаканчике
2. Батончик протеиновый Bombbar фисташковый пломбир
3. Салат с крабовыми палочками «Из Лавки»
4. Кукуруза отварная в початках
5. Энергетический напиток Red Bull
6. Овсяноблин с курицей и томатами Mátes
7. Печенье протеиновое Fit Kit фисташковая кунафа
8. Чипсы протеиновые цельнозерновые Bombbar сладкий чили
9. Пирожное протеиновое ProteinRex брауни вишня
10. Голубцы ленивые запечённые в томатно-сметанном соусе
11. Грудка куриная запечённая 2 шт. «Из Лавки»


## Справочник калорийности

In [4]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

# Define scope
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]

# Load credentials
creds = ServiceAccountCredentials.from_json_keyfile_name('credentials.json', scope)
client = gspread.authorize(creds)

# Open Google Sheet by name or URL
spreadsheet = client.open("Калории")

# Select worksheet/tab by name
worksheet = spreadsheet.worksheet("Reference")

# Get all values as list of rows
data = worksheet.get_all_values()

reference = pd.DataFrame(data[1:], columns=data[0])

reference


,продукт,вес,ккал на 100г,ккал,б на 100г,ж на 100г,у на 100г,б на порцию,ж на порцию,у на порцию
0,Авокадо Хасс «Артфрут» спелое,1,160,160,2,14.7,1.8,2,14.7,1.8
1,Авокадо Хасс Puro Gusto Ready to eat,1,180,180,2,15,8,2,15,8
2,Авока­до Хасс Puro Gusto крупное,2.3,180,414,2,15,8,4.6,34.5,18.4
3,Аджапсандал,2,75.5,151,1.4,6.3,9.3,2.8,12.6,18.6
4,Баклажаны Пармиджано с соусом Болоньезе «Йуми»,2,173,346,5.9,13.1,8,11.8,26.2,16
...,...,...,...,...,...,...,...,...,...,...
116,Яблоки Гренни Смит Отборные «Собрано в саду» 4 шт,8,47,376,0.4,4,9.7,3.2,32,77.6
117,Яблоки Женева,5,47,235,0.4,0.4,9.8,2,2,49
118,Яблоки Малинка,6,59,354,0.2,0.4,15.3,1.2,2.4,91.8
119,Яйца куриные отварные «Из Лавки»,1,164,164,13,12,1,13,12,1


In [5]:
from datetime import datetime
import numpy as np

# Match products with reference records
update = reference[reference['продукт'].isin(titles)]

print("Not found: ", set(list(titles)).difference(set(reference['продукт'])))

# Insert date columns
current_date = datetime.today().strftime('%Y%m%d')
# current_date = '20250717'; print("SETTING manual date")
update.insert(0, 'дата', current_date)
update.insert(len(update.columns), 'активность', np.nan)

# Use Formulas instead of constants
update.loc[:,'ккал'] = 0.0
update.loc[:,'б на порцию'] = 0.0
update.loc[:,'ж на порцию'] = 0.0
update.loc[:,'у на порцию'] = 0.0

# Update types
update.loc[:,'вес'] = update['вес'].astype(float)
update.loc[:,'ккал на 100г'] = update['ккал на 100г'].astype(float)
update.loc[:,'б на 100г'] = update['б на 100г'].astype(float)
update.loc[:,'ж на 100г'] = update['ж на 100г'].astype(float)
update.loc[:,'у на 100г'] = update['у на 100г'].astype(float)

# Compute totals
totals = update.iloc[:,2:].sum()
totals_row = pd.DataFrame([[current_date, 'Total'] + totals.tolist()], columns=update.columns)
update = pd.concat([update, totals_row], ignore_index=True)

# Append the blank row
blank_row = {col:np.nan for col in update.columns}
blank_row['дата']=current_date
update = pd.concat([update, pd.DataFrame([blank_row])], ignore_index=True)

update

Not found:  set()


,дата,продукт,вес,ккал на 100г,ккал,б на 100г,ж на 100г,у на 100г,б на порцию,ж на порцию,у на порцию,активность
0,20250828,Батончик протеиновый Bombbar фисташковый пломбир,0.6,314.0,0.0,33.0,11.0,3.7,0.0,0.0,0.0,NaN
1,20250828,Голубцы ленивые запечённые в томатно-сметанном...,2.5,110.0,0.0,6.5,5.6,8.3,0.0,0.0,0.0,NaN
2,20250828,Грудка куриная запечённая 2 шт. «Из Лавки»,1.4,167.0,0.0,31.0,4.7,0.0,0.0,0.0,0.0,NaN
3,20250828,Кукуруза отварная в початках,4.5,110.0,0.0,2.5,0.5,23.0,0.0,0.0,0.0,NaN
4,20250828,Мороженое протеиновое Bombbar фисташковое без ...,0.8,124.0,0.0,6.3,4.5,14.4,0.0,0.0,0.0,NaN
5,20250828,Овсяноблин с курицей и томатами Mátes,2.1,184.0,0.0,15.3,10.0,8.2,0.0,0.0,0.0,NaN
6,20250828,Печенье протеиновое Fit Kit фисташковая кунафа,0.4,275.0,0.0,27.5,12.5,17.5,0.0,0.0,0.0,NaN
7,20250828,Пирожное протеиновое ProteinRex брауни вишня,0.5,350.0,0.0,11.0,24.0,12.0,0.0,0.0,0.0,NaN
8,20250828,Салат с крабовыми палочками «Из Лавки»,1.8,189.0,0.0,2.5,15.0,11.0,0.0,0.0,0.0,NaN
9,20250828,Чипсы протеиновые цельнозерновые Bombbar сладк...,0.5,271.0,0.0,15.0,3.0,45.0,0.0,0.0,0.0,NaN


In [6]:
from gspread_dataframe import get_as_dataframe, set_with_dataframe

upd_worksheet = spreadsheet.worksheet("2025 New")

existing = get_as_dataframe(upd_worksheet, evaluate_formulas=True, header=0)
first_row = len(existing) + 5

# в первой строке пишем названия колонок, поэтому в формуле надо сдвинуть номер строки
header_shift = 1

# в последних строках total и пустая строка - туда формуклу не пишем
footer_shift = 2

def create_formulas(weight_column, calories_100_column, calories_total_column):
    calories_formula = ['={}{} * {}{}'.format(weight_column, str(x + first_row + header_shift), calories_100_column, str(x + first_row + header_shift)) for x in list(update.index)]
    calories_formula = calories_formula[:len(calories_formula)-footer_shift]
    calories_formula = calories_formula + [''] * footer_shift
    return calories_formula

update['ккал'] = create_formulas(weight_column = 'C', calories_100_column = 'D', calories_total_column = 'ккал')
update['б на порцию'] = create_formulas(weight_column = 'C', calories_100_column = 'F', calories_total_column = 'б на порцию')
update['ж на порцию'] = create_formulas(weight_column = 'C', calories_100_column = 'G', calories_total_column = 'ж на порцию')
update['у на порцию'] = create_formulas(weight_column = 'C', calories_100_column = 'H', calories_total_column = 'у на порцию')

update.columns = [current_date if x=='дата' else x for x in list(update.columns)]

# собираем формулы в Total
total_row_ref = update['продукт']=='Total'
total_row_num = list(update[total_row_ref].index)[0]

update.loc[total_row_num, 'ккал'] = '=SUM(E{}:E{})'.format(first_row + header_shift, first_row + header_shift + total_row_num - 1)
update.loc[total_row_num, 'б на порцию'] = '=SUM(I{}:I{})'.format(first_row + header_shift, first_row + header_shift + total_row_num - 1)
update.loc[total_row_num, 'ж на порцию'] = '=SUM(J{}:J{})'.format(first_row + header_shift, first_row + header_shift + total_row_num - 1)
update.loc[total_row_num, 'у на порцию'] = '=SUM(K{}:K{})'.format(first_row + header_shift, first_row + header_shift + total_row_num - 1)
update.loc[total_row_num, 'ккал на 100г'] = ''
update.loc[total_row_num, 'активность'] = ''

# Step 5: Append new data
set_with_dataframe(
    upd_worksheet, 
    update, 
    row=first_row,
    col=1,
    include_column_header=True)

# Test before use
# worksheet.sort((1, 'desc'))

/var/folders/5w/8klkr2ds68bc0kx8c5_z01t80000gn/T/ipykernel_52930/4014713048.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  update.loc[total_row_num, 'активность'] = ''


In [12]:
worksheet.sort((1, 'des'))

{'spreadsheetId': '1QXHjLqyj_oNjVp8NgGWPWK7-WcvWztRhDVkWKj83hHM',
 'replies': [{}]}

In [11]:
help(worksheet.sort)

Help on method sort in module gspread.worksheet:

sort(*specs: Tuple[int, Literal['asc', 'des']], range: Optional[str] = None) -> MutableMapping[str, Any] method of gspread.worksheet.Worksheet instance
    Sorts worksheet using given sort orders.

    :param list specs: The sort order per column. Each sort order
        represented by a tuple where the first element is a column index
        and the second element is the order itself: 'asc' or 'des'.
    :param str range: The range to sort in A1 notation. By default sorts
        the whole sheet excluding frozen rows.

    Example::

        # Sort sheet A -> Z by column 'B'
        wks.sort((2, 'asc'))

        # Sort range A2:G8 basing on column 'G' A -> Z
        # and column 'B' Z -> A
        wks.sort((7, 'asc'), (2, 'des'), range='A2:G8')

    .. versionadded:: 3.4



In [117]:
import pytesseract
import os
import re
from PIL import Image

def get_most_recent_file(directory):
    
    # List all files    
    files = [os.path.join(directory, f) for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f)) and f.lower().endswith('.png')]

    # Get the most recent
    most_recent_file = max(files, key=os.path.getctime)
    
    return most_recent_file


# Get the screenshot
directory_path = "/Users/konstantin/Documents"
recent_file = get_most_recent_file(directory_path)
print("Found the screenshot file", recent_file)

# Load image
image = Image.open(recent_file)

# Set language to Russian ('rus')
text = pytesseract.image_to_string(image, lang='rus+eng')

UNINFORMATIVE_RE = re.compile(
    r'[\d.,]+\s?(₽|р|Р|P|руб\.?)|Срок годности',
    re.IGNORECASE
)

def extract_product_title(lines):
    pattern = r'^\d+$'
    return sorted([line.strip() for line in lines if not UNINFORMATIVE_RE.search(line) and len(line)>5 and not re.fullmatch(pattern, line.replace(' ','').strip())])

# Example usage
titles = extract_product_title(text.split('\n'))

for i,x in enumerate(titles):
    print(i, ': ', repr(x))


Found the screenshot file /Users/konstantin/Documents/Screenshot 2025-07-20 at 03.39.56.png
0 :  'class GraphState(TypedDict) :'
1 :  'from langchain_core.messages import AnyMessage'
2 :  'from langgraph.graph.message import add_messages'
3 :  'from typing import Annotated'
4 :  'from typing_extensions import TypedDict'
5 :  'messages: Annotated[list[AnyMessage], add_messages ]'


In [2]:
# Manual merging
for pair in [(2, 6)]:
    joined = ' '.join([titles[pair[0]], titles[pair[1]]])
    print("Merged: ",joined)
    titles[pair[0]] = joined
    titles[pair[1]] = ''

titles = [x for x in titles if x != '']

for i,x in enumerate(titles):
    print(i, ': ', x)

Merged:  Мороженое протеиновое Bombbar фисташковое без сахара в вафельном стаканчике
0 :  Батончик протеиновый ProteinRex кокос
1 :  Дыня нарезанная кубиками «Из Лавки»
2 :  Мороженое протеиновое Bombbar фисташковое без сахара в вафельном стаканчике
3 :  Продукт творожный «Даниссимо» с сочным киви 5,5%
4 :  Салат оливье с курицей «Из Лавки»
5 :  Сэндвич стунцом и маринованным луком «Из Лавки»


In [104]:
print(update.head())

       дата                                            продукт  вес  \
0  20250715       Картофель запечёный с ветчиной и сыром Mates  2.2   
1  20250715  Молоко 2,5% «Домик в деревне» ультрапастеризов...  9.5   
2  20250715       Рис жареный самбал с овощами «Старик и море»  1.5   
3  20250715                                  Ролл-фри крабовый  1.8   
4  20250715  Салат из кальмаров командорских и моркови по-к...  1.1   

  ккал на 100г ккал б на 100г ж на 100г у на 100г б на порцию ж на порцию  \
0        133.0  0.0       8.3       5.6      12.3         0.0         0.0   
1         53.0  0.0       2.9       2.5       4.7         0.0         0.0   
2        153.0  0.0       3.7       1.8      30.4         0.0         0.0   
3        236.0  0.0       6.2       7.3      36.4         0.0         0.0   
4        175.0  0.0       6.0      13.0       9.0         0.0         0.0   

  у на порцию  активность  
0         0.0         NaN  
1         0.0         NaN  
2         0.0         NaN 

In [83]:
!pip3.13 install gspread_dataframe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [gspread]1/12 [gspread]uth]]s]ib]
